# FD001 Baseline-Deviation Anomaly Score (engine_id, cycle)

This notebook computes an explainable **0–1 anomaly_score per (engine_id, cycle)** using the **baseline deviation score** approach:

- Baseline per engine = first **N** cycles (default **N=20**)
- Per-sensor z-score vs baseline: `z = (x - mu) / (sigma + eps)`
- Raw anomaly per cycle: `RMS(z)` (or `mean(|z|)`)
- Map raw score to **0–1** globally (fit on TRAIN by default) via **sigmoid** or **min-max**
- Optional smoothing: rolling mean per engine
- Optional explainability: top-K sensors contributing most (largest |z|)

## Required inputs
You need **time-series rows** with at least:
- `engine_id`, `cycle`
- normalized sensor columns (e.g. `s1..s21`, optionally `os1..os3`)

Typical project layout (recommended):

```
jet-cube-turbofan-mvp/
  data/processed/CMAPSS_norm_full/FD001/
    train_FD001_full_norm.csv
    test_FD001_full_norm.csv
  outputs/
    ozcan_fd001_all_rows/           # (optional) RUL predictions + metrics
      predictions_cycle_all_rows.csv
      metrics_cycle_all_rows.json
    anomaly/FD001/
      ... (this notebook will write outputs here)
```

> If your files are in different folders, just update the paths in the next cell.


In [ ]:
# If you run in a fresh environment (Mac / VSCode), install deps once:
# !python -m pip install -U pandas numpy matplotlib


In [ ]:
from pathlib import Path

# ✅ Set this to your repo root (or keep as Path.cwd() if you opened VSCode at repo root)
PROJECT_ROOT = Path("/Users/elifsena/jet-cube-turbofan-mvp")

# === Required (sensor time-series with engine_id + cycle) ===
DATA_DIR = PROJECT_ROOT / "data/processed/CMAPSS_norm_full/FD001"
TRAIN_CSV = DATA_DIR / "train_FD001_full_norm.csv"
TEST_CSV  = DATA_DIR / "test_FD001_full_norm.csv"

# === Optional (join for decision-support) ===
# Must include columns: engine_id, cycle (and ideally y_pred / pred / RUL prediction)
PREDICTIONS_CSV = PROJECT_ROOT / "outputs/ozcan_fd001_all_rows/predictions_cycle_all_rows.csv"

# === Optional (documentation / feature list reference only) ===
METRICS_JSON = PROJECT_ROOT / "outputs/ozcan_fd001_all_rows/metrics_cycle_all_rows.json"

# === Output dir ===
OUT_DIR = PROJECT_ROOT / "outputs/anomaly/FD001"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV, TEST_CSV, PREDICTIONS_CSV, METRICS_JSON, OUT_DIR


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Core column names
ENGINE_ID_COL = "engine_id"
CYCLE_COL = "cycle"

# Baseline definition
BASELINE_N = 20
EPS = 1e-6

# Sensor selection
INCLUDE_OS = False     # include os1/os2/os3 (operating settings) in anomaly calc
SENSOR_PREFIXES = ("s",)  # by default we use s1..s21
OS_COLS = ["os1", "os2", "os3"]  # included only if INCLUDE_OS=True

# Score aggregation
AGG = "rms"            # 'rms' or 'meanabs'
TOPK = 3               # top contributing sensors per row (explainability)

# 0–1 mapping
MAP_MODE = "sigmoid"   # 'sigmoid' or 'minmax'
FIT_MAPPING_ON = "train"  # 'train' or 'all' (train+test)
SIGMOID_K = 1.0        # sigmoid steepness
SMOOTH_WINDOW = 5      # rolling mean window per engine (0/1/None to disable)

def read_csv_safely(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"[{name}] Not found: {path}")
    df = pd.read_csv(path)
    return df

train_df = read_csv_safely(TRAIN_CSV, "TRAIN_CSV")
test_df  = read_csv_safely(TEST_CSV, "TEST_CSV")

# Basic sanity
for name, df in [("TRAIN", train_df), ("TEST", test_df)]:
    missing = [c for c in [ENGINE_ID_COL, CYCLE_COL] if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] Missing required columns: {missing}. Available: {list(df.columns)}")

# Pick sensor columns: numeric columns starting with 's' (and optionally os1..os3)
def pick_sensor_cols(df: pd.DataFrame) -> list[str]:
    cols = []
    for c in df.columns:
        if c in (ENGINE_ID_COL, CYCLE_COL):
            continue
        if c == "RUL":  # don't include label even if present
            continue
        if any(c.startswith(pfx) for pfx in SENSOR_PREFIXES):
            cols.append(c)
    if INCLUDE_OS:
        for c in OS_COLS:
            if c in df.columns:
                cols.append(c)
    # keep only numeric
    cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    return cols

sensor_cols = pick_sensor_cols(train_df)

if not sensor_cols:
    raise ValueError(
        "No sensor columns found automatically. "
        "Either set SENSOR_PREFIXES/INCLUDE_OS or provide sensor_cols manually."
    )

print(f"Chosen sensor cols (n={len(sensor_cols)}): {sensor_cols[:10]}{'...' if len(sensor_cols)>10 else ''}")
print("train shape:", train_df.shape, "| test shape:", test_df.shape)

# Optional: show feature_columns from metrics.json (reference only)
if METRICS_JSON.exists():
    metrics = json.loads(METRICS_JSON.read_text(encoding="utf-8"))
    feat_cols = None
    for keypath in [
        ("all_rows", "feature_columns"),
        ("feature_columns",),
    ]:
        try:
            cur = metrics
            for k in keypath:
                cur = cur[k]
            if isinstance(cur, list):
                feat_cols = cur
                break
        except Exception:
            pass
    if feat_cols is not None:
        print("metrics.json feature_columns (reference):", feat_cols)
        miss = [c for c in feat_cols if c not in train_df.columns]
        if miss:
            print("[WARN] metrics feature_columns missing in TRAIN CSV:", miss)
else:
    print("[INFO] metrics json not found; skipping.")


In [ ]:
def compute_raw_scores_and_topk(
    df: pd.DataFrame,
    sensor_cols: list[str],
    baseline_n: int = 20,
    agg: str = "rms",
    topk: int = 3,
) -> pd.DataFrame:
    df = df.sort_values([ENGINE_ID_COL, CYCLE_COL]).reset_index(drop=True)
    rows = []

    for eid, g in df.groupby(ENGINE_ID_COL, sort=True):
        g = g.sort_values(CYCLE_COL).copy()

        n0 = min(int(baseline_n), int(len(g)))
        base = g.head(n0)
        mu = base[sensor_cols].mean(axis=0)
        sigma = base[sensor_cols].std(axis=0).replace(0.0, np.nan).fillna(0.0) + EPS

        Z = (g[sensor_cols] - mu) / sigma
        Zv = Z.to_numpy(dtype=float)

        if agg == "rms":
            raw = np.sqrt(np.mean(np.square(Zv), axis=1))
        elif agg == "meanabs":
            raw = np.mean(np.abs(Zv), axis=1)
        else:
            raise ValueError("agg must be 'rms' or 'meanabs'")

        absZ = np.abs(Zv)
        k = min(int(topk), int(absZ.shape[1]))
        if k <= 0:
            top_idx = np.empty((len(g), 0), dtype=int)
        else:
            top_idx = np.argpartition(-absZ, kth=k-1, axis=1)[:, :k]

        top_sensors = []
        top_vals = []
        for i in range(len(g)):
            idxs = top_idx[i]
            # sort these top-k by actual value descending
            idxs = idxs[np.argsort(-absZ[i, idxs])] if len(idxs) else idxs
            top_sensors.append([sensor_cols[j] for j in idxs])
            top_vals.append([float(absZ[i, j]) for j in idxs])

        out = pd.DataFrame({
            ENGINE_ID_COL: g[ENGINE_ID_COL].to_numpy(),
            CYCLE_COL: g[CYCLE_COL].to_numpy(),
            "anomaly_raw": raw.astype(float),
            "top_sensors": top_sensors,
            "top_abs_z": top_vals,
        })
        rows.append(out)

    return pd.concat(rows, ignore_index=True)

def fit_mapping_params(raw_scores: np.ndarray, mode: str = "sigmoid", sigmoid_k: float = 1.0) -> dict:
    raw_scores = np.asarray(raw_scores, dtype=float)
    raw_scores = raw_scores[np.isfinite(raw_scores)]
    if len(raw_scores) == 0:
        raise ValueError("No finite raw scores to fit mapping parameters.")
    if mode == "minmax":
        return {"mode": "minmax", "min": float(np.min(raw_scores)), "max": float(np.max(raw_scores))}
    if mode == "sigmoid":
        c = float(np.median(raw_scores))
        return {"mode": "sigmoid", "k": float(sigmoid_k), "c": c}
    raise ValueError("mode must be 'sigmoid' or 'minmax'")

def apply_mapping(raw_scores: np.ndarray, mapping: dict) -> np.ndarray:
    x = np.asarray(raw_scores, dtype=float)
    if mapping["mode"] == "minmax":
        mn, mx = mapping["min"], mapping["max"]
        denom = (mx - mn) if (mx - mn) != 0 else 1.0
        y = (x - mn) / denom
        return np.clip(y, 0.0, 1.0)
    if mapping["mode"] == "sigmoid":
        k, c = mapping["k"], mapping["c"]
        return 1.0 / (1.0 + np.exp(-k * (x - c)))
    raise ValueError("Unknown mapping mode")

def smooth_per_engine(scores_df: pd.DataFrame, window: int = 5) -> np.ndarray:
    if window is None or int(window) <= 1:
        return scores_df["anomaly_score"].to_numpy(dtype=float)
    out = np.empty(len(scores_df), dtype=float)
    # groupby preserves order if we sort first
    for eid, g in scores_df.sort_values([ENGINE_ID_COL, CYCLE_COL]).groupby(ENGINE_ID_COL, sort=True):
        s = g["anomaly_score"].rolling(window=int(window), min_periods=1).mean().to_numpy(dtype=float)
        out[g.index.values] = s
    return out

# --- Compute raw anomaly on train/test
train_scores = compute_raw_scores_and_topk(train_df, sensor_cols, baseline_n=BASELINE_N, agg=AGG, topk=TOPK)
test_scores  = compute_raw_scores_and_topk(test_df,  sensor_cols, baseline_n=BASELINE_N, agg=AGG, topk=TOPK)

# --- Fit mapping params
if FIT_MAPPING_ON == "all":
    mapping = fit_mapping_params(
        np.concatenate([train_scores["anomaly_raw"].values, test_scores["anomaly_raw"].values]),
        mode=MAP_MODE, sigmoid_k=SIGMOID_K
    )
else:
    mapping = fit_mapping_params(train_scores["anomaly_raw"].values, mode=MAP_MODE, sigmoid_k=SIGMOID_K)

train_scores["anomaly_score"] = apply_mapping(train_scores["anomaly_raw"].values, mapping)
test_scores["anomaly_score"]  = apply_mapping(test_scores["anomaly_raw"].values,  mapping)

train_scores["anomaly_score_smoothed"] = smooth_per_engine(train_scores, window=SMOOTH_WINDOW)
test_scores["anomaly_score_smoothed"]  = smooth_per_engine(test_scores,  window=SMOOTH_WINDOW)

print("mapping:", mapping)
train_scores.head()


In [ ]:
# Save outputs
train_out = OUT_DIR / "fd001_anomaly_train.csv"
test_out  = OUT_DIR / "fd001_anomaly_test.csv"

train_scores.to_csv(train_out, index=False)
test_scores.to_csv(test_out, index=False)

print("[OK] wrote:", train_out)
print("[OK] wrote:", test_out)

train_scores.describe(include="all").head()


In [ ]:
# Quick plots (optional)

plt.figure()
plt.hist(train_scores["anomaly_score"].values, bins=50)
plt.title("FD001 anomaly_score distribution (train)")
plt.xlabel("anomaly_score")
plt.ylabel("count")
plt.show()

# Mean anomaly over life progress (0-100%) as a sanity check
def add_life_progress(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["life_progress_pct"] = np.nan
    for eid, g in out.groupby(ENGINE_ID_COL, sort=True):
        idx = g.index.values
        x = g[CYCLE_COL].to_numpy(dtype=float)
        denom = (x.max() - x.min()) if (x.max() - x.min()) != 0 else 1.0
        out.loc[idx, "life_progress_pct"] = 100.0 * (x - x.min()) / denom
    return out

tmp = add_life_progress(test_scores)
bins = np.linspace(0, 100, 101)
tmp["bin"] = np.digitize(tmp["life_progress_pct"].values, bins, right=True)
agg = tmp.groupby("bin")["anomaly_score_smoothed"].median()

plt.figure()
plt.plot(agg.index.values, agg.values)
plt.title("FD001 anomaly_score (median) vs life progress (test)")
plt.xlabel("life progress bin")
plt.ylabel("anomaly_score_smoothed (median)")
plt.show()


In [ ]:
# Optional: join anomaly_score with existing RUL predictions (decision-support table)

if PREDICTIONS_CSV.exists():
    preds = pd.read_csv(PREDICTIONS_CSV)
    need = [ENGINE_ID_COL, CYCLE_COL]
    miss = [c for c in need if c not in preds.columns]
    if miss:
        raise ValueError(f"[PREDICTIONS_CSV] Missing required columns: {miss}. Available: {list(preds.columns)}")

    merged = preds.merge(
        test_scores[[ENGINE_ID_COL, CYCLE_COL, "anomaly_score", "anomaly_score_smoothed", "top_sensors", "top_abs_z"]],
        on=[ENGINE_ID_COL, CYCLE_COL],
        how="left",
    )

    miss_rate = float(merged["anomaly_score"].isna().mean())
    print(f"Join missing anomaly_score rate: {miss_rate:.3%}")

    out_join = OUT_DIR / "fd001_test_with_preds_and_anomaly.csv"
    merged.to_csv(out_join, index=False)
    print("[OK] wrote:", out_join)
    merged.head()
else:
    print("[INFO] predictions csv not found; skip join.")


## Notes for FD002 later

To adapt for FD002:
- update `DATA_DIR`, `TRAIN_CSV`, `TEST_CSV`
- keep `ENGINE_ID_COL` and `CYCLE_COL` names consistent (or change here if FD002 uses different names)
- decide whether to include operating settings (`INCLUDE_OS`)
- keep `BASELINE_N=20` unless your domain requires a different warm-up window
